# Week 14: Final — Demo & Reflection — PHASE 7: Proving Mastery

*Core Mastery: "I can present, defend, and reflect on my engineering work"*

*Computer Programming II | 5 Hours | Dr. Arif Solmaz*

## 🎯 Learning Objectives

1. Complete a self-audit checklist for final project
2. Apply code quality improvements and refactoring techniques
3. Generate a test report with coverage summary
4. Prepare a demo function that showcases the full pipeline
5. Measure and analyze performance of each pipeline stage
6. Reflect on semester learning with structured questions
7. Identify and fix common mistakes in Python data pipelines
8. Preview next-level tools: pandas, scikit-learn, and career paths
9. Write clean, documented, production-ready code
10. Communicate technical decisions effectively

## 🎯 Core Mastery Connection

The final week is about more than just code — it is about demonstrating that you can **present**, **defend**, and **reflect** on your engineering work. You will prepare a demo, review your code quality, analyze performance, and write a semester reflection. These are skills that distinguish a junior engineer from someone ready for professional work.

---
## Part 1: Project Checklist

Before finalizing your project, verify that every deliverable is complete.

| # | Deliverable | Status |
|---|-------------|--------|
| 1 | `read_sensor_data()` works | ☐ |
| 2 | `validate_schema()` catches errors | ☐ |
| 3 | `clean_missing()` removes None rows | ☐ |
| 4 | `remove_invalid()` filters bad status | ☐ |
| 5 | `filter_outliers()` uses IQR | ☐ |
| 6 | `normalize_per_sensor()` scales to [0,1] | ☐ |
| 7 | `compute_summary()` returns dict | ☐ |
| 8 | Time-series plot saved as PNG | ☐ |
| 9 | Histogram saved as PNG | ☐ |
| 10 | Bar chart saved as PNG | ☐ |
| 11 | `moving_average()` implemented | ☐ |
| 12 | `median_filter()` implemented | ☐ |
| 13 | 15+ tests pass | ☐ |
| 14 | Summary report generated | ☐ |
| 15 | `main()` runs end-to-end | ☐ |

**Figure 1.1** — Automated checklist checker

In [ ]:
def check_function_exists(name):
    """Check if a function is defined in the current scope."""
    import builtins
    return name in dir(builtins) or name in globals()

checklist = [
    "read_sensor_data", "validate_schema", "clean_missing",
    "remove_invalid", "filter_outliers", "normalize_per_sensor",
    "compute_summary", "moving_average", "median_filter",
    "generate_report", "group_by_sensor"
]

print("Project Checklist:")
all_ok = True
for func_name in checklist:
    exists = func_name in dir() or func_name in globals()
    status = "✅" if exists else "❌"
    if not exists:
        all_ok = False
    print(f"  {status} {func_name}")

if all_ok:
    print("\n✅ All functions defined!")
else:
    print("\n⚠️ Some functions are missing — define them above or import from previous weeks")

**Figure 1.2** — Self-audit: run all pipeline stages

In [ ]:
import io, csv, math, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- Re-define all pipeline functions for this notebook ----
def read_sensor_data(csv_string):
    reader = csv.DictReader(io.StringIO(csv_string.strip()))
    records = []
    for row in reader:
        try:
            row["value"] = float(row["value"]) if row["value"].strip() else None
        except ValueError:
            row["value"] = None
        records.append(row)
    return records

def validate_schema(records):
    required = {"timestamp", "sensor_id", "value", "status"}
    if not records: raise ValueError("No records")
    missing = required - set(records[0].keys())
    if missing: raise ValueError(f"Missing: {missing}")
    for r in records:
        if not r.get("status") or r["status"].strip() == "": r["status"] = "UNKNOWN"
    return records

def clean_missing(records): return [r for r in records if r["value"] is not None]
def remove_invalid(records): return [r for r in records if r["status"] in {"OK","WARN","ERROR"}]

def filter_outliers(records, threshold=1.5):
    values = sorted(r["value"] for r in records)
    n = len(values)
    q1, q3 = values[n//4], values[3*n//4]
    iqr = q3 - q1
    lo, hi = q1 - threshold*iqr, q3 + threshold*iqr
    return [r for r in records if lo <= r["value"] <= hi]

def group_by_sensor(records):
    g = {}
    for r in records:
        g.setdefault(r["sensor_id"], []).append(r)
    return g

def normalize_per_sensor(records):
    groups = group_by_sensor(records)
    result = []
    for sid in sorted(groups):
        vals = [r["value"] for r in groups[sid] if r["value"] is not None]
        if not vals: continue
        vmin, vmax = min(vals), max(vals)
        rng = vmax - vmin if vmax != vmin else 1.0
        for r in groups[sid]:
            if r["value"] is not None:
                rc = dict(r); rc["value_raw"] = r["value"]
                rc["value"] = round((r["value"] - vmin) / rng, 6)
                result.append(rc)
    return result

def compute_summary(records, key="value_raw"):
    groups = group_by_sensor(records)
    summary = {}
    for sid in sorted(groups):
        vals = [r.get(key, r["value"]) for r in groups[sid]]
        n = len(vals); mean = sum(vals)/n
        var = sum((v-mean)**2 for v in vals)/n
        summary[sid] = {"mean":round(mean,2),"std":round(math.sqrt(var),2),
                        "min":round(min(vals),2),"max":round(max(vals),2),"count":n}
    return summary

def moving_average(data, window):
    result = []; half = window // 2
    for i in range(len(data)):
        s, e = max(0,i-half), min(len(data),i+half+1)
        result.append(round(sum(data[s:e])/(e-s), 4))
    return result

def median_filter(data, window):
    result = []; half = window // 2
    for i in range(len(data)):
        s, e = max(0,i-half), min(len(data),i+half+1)
        chunk = sorted(data[s:e]); mid = len(chunk)//2
        result.append(chunk[mid] if len(chunk)%2 else (chunk[mid-1]+chunk[mid])/2)
    return result

def generate_report(summary, config):
    lines = ["="*50, "SENSOR LOG ANALYZER REPORT", "="*50]
    for sid in sorted(summary): 
        s = summary[sid]
        lines.append(f"{sid}: mean={s['mean']}, std={s['std']}, min={s['min']}, max={s['max']}, n={s['count']}")
    lines.append("="*50)
    return "\n".join(lines)

SAMPLE_CSV = """timestamp,sensor_id,value,status
2025-06-01 08:00,S01,23.5,OK
2025-06-01 08:00,S02,45.1,OK
2025-06-01 08:00,S03,12.8,OK
2025-06-01 09:00,S01,24.1,OK
2025-06-01 09:00,S02,,WARN
2025-06-01 09:00,S03,13.2,OK
2025-06-01 10:00,S01,9999,ERROR
2025-06-01 10:00,S02,47.3,OK
2025-06-01 10:00,S03,11.9,OK
2025-06-01 11:00,S01,25.0,OK
2025-06-01 11:00,S02,44.8,OK
2025-06-01 11:00,S03,,
2025-06-01 12:00,S01,23.8,OK
2025-06-01 12:00,S02,46.5,WARN
2025-06-01 12:00,S03,14.1,OK
2025-06-01 13:00,S01,24.9,OK
2025-06-01 13:00,S02,9999,OK
2025-06-01 13:00,S03,13.7,OK
2025-06-01 14:00,S01,22.1,INVALID
2025-06-01 14:00,S02,43.2,OK
2025-06-01 14:00,S03,15.0,OK
2025-06-01 15:00,S01,26.3,OK
2025-06-01 15:00,S02,48.0,OK
2025-06-01 15:00,S03,12.5,OK"""

print("Self-audit: running full pipeline...")
data = read_sensor_data(SAMPLE_CSV)
data = validate_schema(data)
data = clean_missing(data)
data = remove_invalid(data)
data = filter_outliers(data)
normed = normalize_per_sensor(data)
stats = compute_summary(normed)
print(f"Pipeline ran: {len(normed)} records, {len(stats)} sensors")
print("Self-audit PASSED!")

---
## Part 2: Code Quality Review

Good code quality means:
- Clear function names and docstrings
- No magic numbers (use CONFIG)
- DRY — Don't Repeat Yourself
- Consistent naming (snake_case)
- Error handling with meaningful messages

| Code Smell | Fix |
|-----------|-----|
| Magic numbers | Replace with named constants |
| Repeated code | Extract into functions |
| Missing docstrings | Add to every function |
| Long functions (>20 lines) | Break into smaller ones |
| No error handling | Add try/except |

**Figure 2.1** — Code smell detector

In [ ]:
import inspect

def check_code_quality(func):
    """Analyze a function for common code quality issues."""
    source = inspect.getsource(func)
    lines = source.split("\n")
    issues = []
    # Check for docstring
    if '"""\' not in source and "\'\'\'" not in source:
        issues.append("Missing docstring")
    # Check length
    if len(lines) > 25:
        issues.append(f"Function too long ({len(lines)} lines)")
    # Check for magic numbers
    import re
    numbers = re.findall(r"(?<![\"\'])\b\d+\.\d+\b", source)
    if len(numbers) > 2:
        issues.append(f"Possible magic numbers: {numbers[:3]}")
    return issues

functions_to_check = [read_sensor_data, validate_schema, clean_missing,
                      filter_outliers, normalize_per_sensor, compute_summary]
print("Code Quality Report:")
for func in functions_to_check:
    issues = check_code_quality(func)
    status = "✅" if not issues else "⚠️"
    print(f"  {status} {func.__name__}: {issues if issues else 'Clean'}")

**Figure 2.2** — Refactoring example: extracting repeated grouping

In [ ]:
# BEFORE: repeated grouping logic in multiple functions
# AFTER: use group_by_sensor() everywhere

def normalize_per_sensor_v2(records):
    """Refactored: uses group_by_sensor helper."""
    groups = group_by_sensor(records)  # DRY!
    result = []
    for sid in sorted(groups):
        vals = [r["value"] for r in groups[sid] if r["value"] is not None]
        if not vals: continue
        vmin, vmax = min(vals), max(vals)
        rng = vmax - vmin if vmax != vmin else 1.0
        for r in groups[sid]:
            if r["value"] is not None:
                rc = dict(r)
                rc["value_raw"] = r["value"]
                rc["value"] = round((r["value"] - vmin) / rng, 6)
                result.append(rc)
    return result

# Verify refactored version produces same output
original = normalize_per_sensor(data)
refactored = normalize_per_sensor_v2(data)
assert len(original) == len(refactored), "Refactored version differs!"
print(f"Refactored normalize_per_sensor_v2 verified: {len(refactored)} records")

**Figure 2.3** — Adding proper error handling

In [ ]:
def safe_read_sensor_data(csv_string):
    """Read sensor data with comprehensive error handling."""
    if not csv_string or not csv_string.strip():
        raise ValueError("CSV string is empty")
    try:
        reader = csv.DictReader(io.StringIO(csv_string.strip()))
        records = []
        for i, row in enumerate(reader):
            try:
                row["value"] = float(row["value"]) if row.get("value", "").strip() else None
            except (ValueError, AttributeError):
                row["value"] = None
            records.append(row)
        if not records:
            raise ValueError("CSV parsed but no data rows found")
        return records
    except csv.Error as e:
        raise ValueError(f"CSV parsing error: {e}")

# Test error handling
try: safe_read_sensor_data("")
except ValueError as e: print(f"Caught: {e}")
try: safe_read_sensor_data("bad,csv\nno,header")
except Exception as e: print(f"Caught: {e}")
print("Error handling works correctly")

---
## Part 3: Test Report

A test report summarizes what was tested, how many tests passed, and what edge cases were covered.

**Figure 3.1** — Test runner with reporting

In [ ]:
def run_test(name, test_func):
    """Run a single test and return (name, passed, error)."""
    try:
        test_func()
        return (name, True, None)
    except AssertionError as e:
        return (name, False, str(e))
    except Exception as e:
        return (name, False, f"Unexpected: {e}")

def test_read():
    records = read_sensor_data(SAMPLE_CSV)
    assert len(records) == 24
    assert all("timestamp" in r for r in records)

def test_clean_missing():
    records = read_sensor_data(SAMPLE_CSV)
    records = validate_schema(records)
    cleaned = clean_missing(records)
    assert all(r["value"] is not None for r in cleaned)

def test_clean_invalid():
    records = read_sensor_data(SAMPLE_CSV)
    records = validate_schema(records)
    cleaned = remove_invalid(records)
    assert all(r["status"] in {"OK","WARN","ERROR"} for r in cleaned)

def test_outliers():
    records = read_sensor_data(SAMPLE_CSV)
    records = validate_schema(records)
    records = clean_missing(records)
    records = remove_invalid(records)
    filtered = filter_outliers(records)
    assert len(filtered) < len(records) or len(filtered) == len(records)

def test_normalize_bounds():
    records = read_sensor_data(SAMPLE_CSV)
    records = validate_schema(records)
    records = clean_missing(records)
    records = remove_invalid(records)
    records = filter_outliers(records)
    normed = normalize_per_sensor(records)
    vals = [r["value"] for r in normed]
    assert all(0 <= v <= 1 for v in vals)

tests = [
    ("test_read", test_read),
    ("test_clean_missing", test_clean_missing),
    ("test_clean_invalid", test_clean_invalid),
    ("test_outliers", test_outliers),
    ("test_normalize_bounds", test_normalize_bounds),
]

results = [run_test(n, f) for n, f in tests]
passed = sum(1 for _, p, _ in results if p)
print(f"Test Report: {passed}/{len(results)} passed")
for name, ok, err in results:
    print(f"  {'✅' if ok else '❌'} {name}" + (f" — {err}" if err else ""))

**Figure 3.2** — Edge case tests

In [ ]:
def test_empty_csv():
    try:
        read_sensor_data("")
    except:
        pass  # Expected

def test_single_record():
    csv = "timestamp,sensor_id,value,status\n2025-01-01,S01,10.0,OK"
    records = read_sensor_data(csv)
    assert len(records) == 1
    normed = normalize_per_sensor(records)
    assert len(normed) == 1

def test_all_same_value():
    csv = "timestamp,sensor_id,value,status\n"
    csv += "\n".join(f"2025-01-0{i+1},S01,42.0,OK" for i in range(5))
    records = read_sensor_data(csv)
    normed = normalize_per_sensor(records)
    # All same value, should all normalize to 0
    assert all(r["value"] == 0.0 for r in normed)

def test_ma_single():
    assert moving_average([5], 3) == [5]

def test_median_single():
    assert median_filter([5], 3) == [5]

edge_tests = [
    ("test_empty_csv", test_empty_csv),
    ("test_single_record", test_single_record),
    ("test_all_same_value", test_all_same_value),
    ("test_ma_single", test_ma_single),
    ("test_median_single", test_median_single),
]

edge_results = [run_test(n, f) for n, f in edge_tests]
edge_passed = sum(1 for _, p, _ in edge_results if p)
print(f"Edge Case Tests: {edge_passed}/{len(edge_results)} passed")
for name, ok, err in edge_results:
    print(f"  {'✅' if ok else '❌'} {name}" + (f" — {err}" if err else ""))

**Figure 3.3** — Full coverage summary

In [ ]:
all_results = results + edge_results
total = len(all_results)
total_passed = sum(1 for _, p, _ in all_results if p)

print("=" * 40)
print("FULL TEST COVERAGE SUMMARY")
print("=" * 40)
print(f"Total tests:  {total}")
print(f"Passed:       {total_passed}")
print(f"Failed:       {total - total_passed}")
print(f"Pass rate:    {total_passed/total*100:.0f}%")
print("=" * 40)

coverage_areas = [
    "Reading CSV data",
    "Schema validation",
    "Missing value cleaning",
    "Invalid status removal",
    "Outlier filtering (IQR)",
    "Per-sensor normalization",
    "Moving average filter",
    "Median filter",
    "Edge cases (empty, single, identical)"
]
print("\nAreas covered:")
for area in coverage_areas:
    print(f"  ✅ {area}")

---
## Part 4: Demo Preparation

A demo function showcases the full pipeline with clear output at each stage.

**Figure 4.1** — `demo_pipeline()` function

In [ ]:
def demo_pipeline(csv_string):
    """Demonstrate the full pipeline with stage-by-stage output."""
    print("🔬 SENSOR LOG ANALYZER — DEMO")
    print("=" * 50)

    print("\n📥 Stage 1: Reading data...")
    records = read_sensor_data(csv_string)
    print(f"   Loaded {len(records)} records")
    print(f"   Columns: {list(records[0].keys())}")

    print("\n✅ Stage 2: Validating schema...")
    records = validate_schema(records)
    unknown = sum(1 for r in records if r["status"] == "UNKNOWN")
    print(f"   {unknown} records set to UNKNOWN status")

    print("\n🧹 Stage 3: Cleaning...")
    before = len(records)
    records = clean_missing(records)
    print(f"   Missing removed: {before} → {len(records)}")
    before = len(records)
    records = remove_invalid(records)
    print(f"   Invalid removed: {before} → {len(records)}")
    before = len(records)
    records = filter_outliers(records)
    print(f"   Outliers removed: {before} → {len(records)}")

    print("\n📊 Stage 4: Normalizing...")
    normed = normalize_per_sensor(records)
    groups = group_by_sensor(normed)
    for sid in sorted(groups):
        vals = [r["value"] for r in groups[sid]]
        print(f"   {sid}: [{min(vals):.3f}, {max(vals):.3f}]")

    print("\n📈 Stage 5: Summary statistics...")
    summary = compute_summary(normed)
    for sid, s in sorted(summary.items()):
        print(f"   {sid}: mean={s['mean']}, std={s['std']}")

    print("\n" + "=" * 50)
    print("🎯 DEMO COMPLETE")
    return normed, summary

normed, summary = demo_pipeline(SAMPLE_CSV)

**Figure 4.2** — Demo with visualization

In [ ]:
def demo_with_plots(normed, summary):
    """Create demo plots."""
    # Time series
    groups = group_by_sensor(normed)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Plot 1: time series
    for sid in sorted(groups):
        vals = [r.get("value_raw", r["value"]) for r in groups[sid]]
        axes[0].plot(vals, "o-", label=sid)
    axes[0].set_title("Time Series")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot 2: histogram
    all_vals = [r.get("value_raw", r["value"]) for r in normed]
    axes[1].hist(all_vals, bins=12, edgecolor="black", alpha=0.7)
    axes[1].set_title("Value Distribution")
    axes[1].grid(True, alpha=0.3)

    # Plot 3: bar chart
    sids = sorted(summary.keys())
    means = [summary[s]["mean"] for s in sids]
    axes[2].bar(sids, means, color=["#e74c3c","#3498db","#2ecc71"], edgecolor="black")
    axes[2].set_title("Mean per Sensor")
    axes[2].grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    print("Demo plots created (3 subplots)")
    return fig

fig = demo_with_plots(normed, summary)

---
## Part 5: Performance Analysis

Timing each stage helps identify bottlenecks.

**Figure 5.1** — Stage timer

In [ ]:
import time

def timed_pipeline(csv_string, n_runs=10):
    """Time each pipeline stage over n_runs."""
    timings = {}
    for stage_name, func, needs_threshold in [
        ("read", lambda d: read_sensor_data(d), False),
    ]:
        pass  # We need a different approach for chained stages

    # Chain approach
    stages = [
        ("read", lambda: read_sensor_data(csv_string)),
    ]
    data_store = {}

    t0 = time.time()
    for _ in range(n_runs):
        data_store["read"] = read_sensor_data(csv_string)
    timings["read"] = (time.time() - t0) / n_runs

    t0 = time.time()
    for _ in range(n_runs):
        data_store["validate"] = validate_schema(list(data_store["read"]))
    timings["validate"] = (time.time() - t0) / n_runs

    t0 = time.time()
    for _ in range(n_runs):
        d = clean_missing(data_store["validate"])
        d = remove_invalid(d)
        data_store["clean"] = filter_outliers(d)
    timings["clean"] = (time.time() - t0) / n_runs

    t0 = time.time()
    for _ in range(n_runs):
        data_store["normalize"] = normalize_per_sensor(data_store["clean"])
    timings["normalize"] = (time.time() - t0) / n_runs

    t0 = time.time()
    for _ in range(n_runs):
        data_store["summary"] = compute_summary(data_store["normalize"])
    timings["summary"] = (time.time() - t0) / n_runs

    return timings

timings = timed_pipeline(SAMPLE_CSV)
print(f"{'Stage':>12} {'Time (ms)':>10}")
print("-" * 24)
for stage, t in timings.items():
    print(f"{stage:>12} {t*1000:10.3f}")
total = sum(timings.values())
print(f"{'TOTAL':>12} {total*1000:10.3f}")

**Figure 5.2** — Identifying the bottleneck

In [ ]:
slowest = max(timings, key=timings.get)
fastest = min(timings, key=timings.get)
print(f"Slowest stage: {slowest} ({timings[slowest]*1000:.3f} ms)")
print(f"Fastest stage: {fastest} ({timings[fastest]*1000:.3f} ms)")
print(f"Ratio: {timings[slowest]/timings[fastest]:.1f}x")
print(f"\nOptimization tip: Focus on the '{slowest}' stage for performance gains.")

**Figure 5.3** — Performance bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
stage_names = list(timings.keys())
stage_times = [t * 1000 for t in timings.values()]
bars = ax.barh(stage_names, stage_times, color="steelblue", edgecolor="black")
ax.set_xlabel("Time (ms)")
ax.set_title("Pipeline Stage Performance")
for bar, t in zip(bars, stage_times):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{t:.2f} ms", va="center", fontsize=9)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
print("Performance chart created")

---
## Part 6: Semester Reflection

Structured reflection helps solidify learning. Answer each question thoughtfully.

**Figure 6.1** — Reflection questions

In [ ]:
reflection_questions = [
    "1. What was the most challenging concept you learned this semester?",
    "2. Which project or exercise are you most proud of?",
    "3. How has your approach to debugging changed since Week 1?",
    "4. What would you do differently if you started the course again?",
    "5. How confident are you in writing a complete Python program from scratch?",
    "6. Which topic do you want to explore further after this course?",
    "7. How did working with data pipelines change your understanding of programming?",
    "8. What advice would you give to next semester's students?",
]

print("Semester Reflection Questions:")
print("=" * 50)
for q in reflection_questions:
    print(f"\n{q}")
    print("  Your answer: ___________________________________")

**Figure 6.2** — Skill self-assessment

In [ ]:
skills = {
    "Variables & Types": 0,
    "Control Flow (if/for/while)": 0,
    "Functions": 0,
    "File I/O": 0,
    "Error Handling": 0,
    "Data Structures (list/dict)": 0,
    "Testing with assert": 0,
    "Visualization (matplotlib)": 0,
    "Data Cleaning": 0,
    "Code Refactoring": 0,
}

print("Skill Self-Assessment (rate 1-5):")
print("=" * 40)
for skill, score in skills.items():
    bar = "█" * score + "░" * (5 - score)
    print(f"  {skill:30s} [{bar}] {score}/5")
print("\n📝 Update the scores above to reflect your confidence level!")

---
## Part 7: Common Mistakes

Top 10 issues found in student projects, with examples and fixes.

| # | Mistake | Impact |
|---|---------|--------|
| 1 | Not handling empty input | Crashes |
| 2 | Mutating input data | Side effects |
| 3 | Missing edge case tests | Hidden bugs |
| 4 | Hard-coded file paths | Not portable |
| 5 | No docstrings | Unreadable code |
| 6 | Ignoring return values | Lost data |
| 7 | Division by zero in stats | Crashes |
| 8 | Not closing files | Resource leak |
| 9 | Incorrect IQR calculation | Wrong outliers |
| 10 | Plotting without labels | Unclear charts |

**Figure 7.1** — Mistake: mutating input

In [ ]:
# BAD: mutates the original list
def bad_clean(records):
    for r in records:
        if r["value"] is None:
            records.remove(r)  # Mutates while iterating!
    return records

# GOOD: creates a new list
def good_clean(records):
    return [r for r in records if r["value"] is not None]

# Demonstrate the problem
test_data = [{"value": 1}, {"value": None}, {"value": None}, {"value": 2}]
result_bad = bad_clean(list(test_data))  # copy to avoid permanent damage
result_good = good_clean(test_data)
print(f"Bad clean result:  {len(result_bad)} records (missed some!)")
print(f"Good clean result: {len(result_good)} records (correct!)")

**Figure 7.2** — Mistake: division by zero in normalization

In [ ]:
# BAD: crashes when all values are the same
def bad_normalize(values):
    vmin, vmax = min(values), max(values)
    return [(v - vmin) / (vmax - vmin) for v in values]  # ZeroDivisionError!

# GOOD: handles edge case
def good_normalize(values):
    vmin, vmax = min(values), max(values)
    rng = vmax - vmin if vmax != vmin else 1.0
    return [(v - vmin) / rng for v in values]

try:
    bad_normalize([5, 5, 5])
    print("Bad normalize: no error (unexpected)")
except ZeroDivisionError:
    print("Bad normalize: ZeroDivisionError (expected!)")

result = good_normalize([5, 5, 5])
print(f"Good normalize: {result} (handles edge case)")

**Figure 7.3** — Mistake: plotting without labels

In [ ]:
# BAD plot — no title, labels, or legend
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot([1,2,3], [10,20,15])
ax1.set_title("BAD: No labels")

# GOOD plot — complete with all elements
ax2.plot([1,2,3], [10,20,15], "o-", label="Sensor S01", color="steelblue")
ax2.set_title("GOOD: Proper Labels")
ax2.set_xlabel("Time Index")
ax2.set_ylabel("Value")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
print("Comparison: bad vs good plotting practices")

---
## Part 8: Next Steps

A preview of tools and paths beyond this course.

**Figure 8.1** — pandas preview

In [ ]:
# pandas makes data pipelines much simpler
# (This is a preview — pandas is not required for this course)

try:
    import pandas as pd
    df = pd.read_csv(io.StringIO(SAMPLE_CSV.strip()))
    print("pandas DataFrame:")
    print(df.head())
    print(f"\nShape: {df.shape}")
    print(f"\nDescribe:\n{df.describe()}")
except ImportError:
    print("pandas is not installed — this is just a preview")
    print("Install with: pip install pandas")
    print("\nWith pandas, the entire cleaning pipeline becomes:")
    print("  df = pd.read_csv('data.csv')")
    print("  df = df.dropna()")
    print("  df = df[df['status'].isin(['OK','WARN','ERROR'])]")
    print("  df['value_norm'] = df.groupby('sensor_id')['value'].transform(")
    print("      lambda x: (x - x.min()) / (x.max() - x.min()))")

**Figure 8.2** — Career paths in Python

In [ ]:
career_paths = {
    "Data Engineering": [
        "pandas, NumPy, Apache Spark",
        "ETL pipelines, databases (SQL)",
        "Cloud platforms (AWS, GCP, Azure)"
    ],
    "Data Science": [
        "scikit-learn, TensorFlow, PyTorch",
        "Statistics, machine learning",
        "Jupyter notebooks, visualization"
    ],
    "Web Development": [
        "Django, Flask, FastAPI",
        "REST APIs, databases",
        "HTML/CSS/JavaScript"
    ],
    "Embedded / IoT": [
        "MicroPython, Raspberry Pi",
        "Sensor integration, real-time data",
        "C/C++ alongside Python"
    ],
    "DevOps / Automation": [
        "Docker, Kubernetes",
        "CI/CD pipelines",
        "Scripting and automation"
    ]
}

print("Python Career Paths:")
print("=" * 50)
for path, skills in career_paths.items():
    print(f"\n🎯 {path}:")
    for skill in skills:
        print(f"   • {skill}")

---
## Exercises

Complete each exercise in the code cell below it.

**EX1:** Write `count_functions(module_source)` that counts function definitions (`def ...`) in a source string.

In [ ]:
# ✏️ [EX1]
def count_functions(module_source):
    pass


**EX2:** Write `check_docstrings(func_list)` that returns a list of function names missing docstrings.

In [ ]:
# ✏️ [EX2]
def check_docstrings(func_list):
    pass


**EX3:** Write `measure_time(func, *args)` that returns `(result, elapsed_seconds)`.

In [ ]:
# ✏️ [EX3]
def measure_time(func, *args):
    pass


**EX4:** Write `generate_test_data(n_sensors, n_readings)` that creates synthetic CSV data.

In [ ]:
# ✏️ [EX4]
def generate_test_data(n_sensors=3, n_readings=50):
    pass


**EX5:** Write `pipeline_diff(data1, data2)` that reports differences between two lists of records.

In [ ]:
# ✏️ [EX5]
def pipeline_diff(data1, data2):
    pass


**EX6:** Write `code_line_count(source)` that counts non-empty, non-comment lines.

In [ ]:
# ✏️ [EX6]
def code_line_count(source):
    pass


**EX7:** Write 3 reflection answers as print statements: proudest moment, biggest challenge, key takeaway.

In [ ]:
# ✏️ [EX7]
print("Proudest moment: ")
print("Biggest challenge: ")
print("Key takeaway: ")


**EX8:** Write `suggest_improvements(summary)` that analyzes summary stats and prints 3 actionable suggestions.

In [ ]:
# ✏️ [EX8]
def suggest_improvements(summary):
    pass


**EX9:** Write `format_duration(seconds)` that converts seconds to `'Xm Ys'` format.

In [ ]:
# ✏️ [EX9]
def format_duration(seconds):
    pass


**EX10:** Write `create_project_zip(file_list, zip_name)` that creates a zip archive of given files.

In [ ]:
# ✏️ [EX10]
def create_project_zip(file_list, zip_name="project.zip"):
    pass


**EX11:** Write `validate_pipeline_output(output_dir)` that checks all expected files exist in the output directory.

In [ ]:
# ✏️ [EX11]
def validate_pipeline_output(output_dir):
    pass


**EX12:** Write `compare_implementations(func1, func2, test_input)` that compares two function results and timing.

In [ ]:
# ✏️ [EX12]
def compare_implementations(func1, func2, test_input):
    pass


**EX13:** Write `grade_checklist(checklist_results)` that takes a dict of `{item: bool}` and prints a score.

In [ ]:
# ✏️ [EX13]
def grade_checklist(checklist_results):
    pass


**EX14:** Write `semester_stats()` that prints: total exercises attempted, total functions written, total tests run.

In [ ]:
# ✏️ [EX14]
def semester_stats():
    pass


**EX15:** Write a `final_message()` that prints a personalized course completion message using STUDENT_NAME.

In [ ]:
# ✏️ [EX15]
def final_message():
    pass


---
### 🎓 Course Reflection

Congratulations on completing Computer Programming II! Over 14 weeks you have progressed from basic Python syntax to building a complete data pipeline with reading, validation, cleaning, normalization, statistics, visualization, testing, and export. You have learned to write clean functions, handle errors, test systematically, and present your work.

Remember: programming is a skill that improves with practice. Keep building projects, keep testing your code, and keep learning. The foundations you have built here will serve you throughout your engineering career.

*"The only way to learn a new programming language is by writing programs in it." — Dennis Ritchie*